# Exam 01: Complete Solution Notebook

**Course:** Machine Learning Theory  
**Institution:** Universidad Nacional de Colombia  
**Term:** 2026-1  
**Deliverable language:** English

This notebook is the complete solution deliverable for Exam 01. It contains:

- the mathematical derivations,
- the implementation used to verify the derivations,
- the numerical outputs required by the exam.

The original statement is available as `main.pdf` in the same folder.


## Setup

The next code cell defines all helper functions used in the solution. The notebook is self-contained: it does not import any local exam helper modules.


In [1]:
from __future__ import annotations

import argparse
import gzip
import json
import shutil
import urllib.request
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np


ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "exams" / "01" / "main.pdf").exists():
    ROOT = ROOT.parent
if not (ROOT / "exams" / "01" / "main.pdf").exists():
    raise FileNotFoundError("Could not locate exams/01/main.pdf from the current working directory.")
REPO_ROOT = ROOT
DEFAULT_OUTPUT_PATH = REPO_ROOT / "exams" / "01" / "solutions" / "artifacts" / "numeric_results.json"
MNIST_DATA_DIR = REPO_ROOT / "data" / "raw" / "mnist"
MNIST_BASE_URLS = (
    "https://storage.googleapis.com/cvdf-datasets/mnist/",
    "http://yann.lecun.com/exdb/mnist/",
)


@dataclass(frozen=True)
class PairScore:
    digit_a: int
    digit_b: int
    mmd2_biased: float


def build_polynomial_design_matrix(x_samples: np.ndarray) -> np.ndarray:
    """Builds the design matrix for phi(x) = [1, x, x^2]^T."""
    x_vector = np.asarray(x_samples, dtype=np.float64).reshape(-1)
    return np.column_stack(
        (
            np.ones_like(x_vector),
            x_vector,
            x_vector**2,
        )
    )


def solve_regularized_least_squares(
    design_matrix: np.ndarray,
    targets: np.ndarray,
    lambda_reg: float,
) -> np.ndarray:
    """Solves (Phi^T Phi + lambda I) w = Phi^T t."""
    if lambda_reg < 0.0:
        raise ValueError("lambda_reg must be non-negative.")

    phi = np.asarray(design_matrix, dtype=np.float64)
    t = np.asarray(targets, dtype=np.float64).reshape(-1)
    gram = phi.T @ phi
    rhs = phi.T @ t
    system = gram + lambda_reg * np.eye(phi.shape[1], dtype=np.float64)
    return np.linalg.solve(system, rhs)


def solve_kernelized_regularized_least_squares(
    design_matrix: np.ndarray,
    targets: np.ndarray,
    lambda_reg: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Solves the dual system and maps it back to the primal weights."""
    if lambda_reg < 0.0:
        raise ValueError("lambda_reg must be non-negative.")

    phi = np.asarray(design_matrix, dtype=np.float64)
    t = np.asarray(targets, dtype=np.float64).reshape(-1)
    kernel_matrix = phi @ phi.T
    alpha = np.linalg.solve(
        kernel_matrix + lambda_reg * np.eye(phi.shape[0], dtype=np.float64),
        t,
    )
    return alpha, phi.T @ alpha


def pairwise_squared_distances(x_matrix: np.ndarray) -> np.ndarray:
    """Computes the full squared Euclidean distance matrix."""
    x = np.asarray(x_matrix, dtype=np.float64)
    squared_norms = np.sum(x * x, axis=1, keepdims=True)
    distances = squared_norms + squared_norms.T - 2.0 * (x @ x.T)
    return np.maximum(distances, 0.0)


def rbf_kernel_matrix(x_matrix: np.ndarray, gamma: float) -> np.ndarray:
    """Builds the Gaussian/RBF Gram matrix."""
    if gamma <= 0.0:
        raise ValueError("gamma must be strictly positive.")
    return np.exp(-gamma * pairwise_squared_distances(x_matrix))


def cross_rbf_kernel_matrix(x_matrix: np.ndarray, y_matrix: np.ndarray, gamma: float) -> np.ndarray:
    """Builds the cross-kernel matrix between two sample sets."""
    if gamma <= 0.0:
        raise ValueError("gamma must be strictly positive.")

    x = np.asarray(x_matrix, dtype=np.float64)
    y = np.asarray(y_matrix, dtype=np.float64)
    x_norms = np.sum(x * x, axis=1, keepdims=True)
    y_norms = np.sum(y * y, axis=1, keepdims=True)
    distances = x_norms + y_norms.T - 2.0 * (x @ y.T)
    distances = np.maximum(distances, 0.0)
    return np.exp(-gamma * distances)


def biased_mmd2_from_gram(k_xx: np.ndarray, k_yy: np.ndarray, k_xy: np.ndarray) -> float:
    """Returns the biased empirical MMD^2 estimate."""
    m = k_xx.shape[0]
    n = k_yy.shape[0]
    return float(
        np.sum(k_xx) / (m * m)
        + np.sum(k_yy) / (n * n)
        - 2.0 * np.sum(k_xy) / (m * n)
    )


def median_heuristic_gamma(x_matrix: np.ndarray) -> float:
    """Chooses gamma = 1 / median(nonzero squared distances)."""
    distances = pairwise_squared_distances(x_matrix)
    nonzero = distances[distances > 0.0]
    if nonzero.size == 0:
        raise ValueError("Median heuristic requires at least two distinct samples.")
    return float(1.0 / np.median(nonzero))


def ensure_mnist_file(filename: str) -> Path:
    """Downloads one MNIST gzip file into data/raw/mnist if missing."""
    destination = MNIST_DATA_DIR / filename
    if destination.exists():
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error: Exception | None = None

    for base_url in MNIST_BASE_URLS:
        try:
            with urllib.request.urlopen(base_url + filename, timeout=60) as response:
                with destination.open("wb") as output_handle:
                    shutil.copyfileobj(response, output_handle)
            return destination
        except Exception as exc:  # pragma: no cover - network fallback path
            last_error = exc
            if destination.exists():
                destination.unlink()

    raise RuntimeError(f"Failed to download MNIST file {filename!r}.") from last_error


def load_mnist_images(path: Path) -> np.ndarray:
    """Loads an IDX image file compressed with gzip."""
    with gzip.open(path, "rb") as handle:
        magic = int.from_bytes(handle.read(4), "big")
        if magic != 2051:
            raise ValueError(f"Unexpected image magic number {magic}.")
        count = int.from_bytes(handle.read(4), "big")
        rows = int.from_bytes(handle.read(4), "big")
        cols = int.from_bytes(handle.read(4), "big")
        data = np.frombuffer(handle.read(), dtype=np.uint8)
    return data.reshape(count, rows * cols).astype(np.float64) / 255.0


def load_mnist_labels(path: Path) -> np.ndarray:
    """Loads an IDX label file compressed with gzip."""
    with gzip.open(path, "rb") as handle:
        magic = int.from_bytes(handle.read(4), "big")
        if magic != 2049:
            raise ValueError(f"Unexpected label magic number {magic}.")
        count = int.from_bytes(handle.read(4), "big")
        data = np.frombuffer(handle.read(), dtype=np.uint8)
    return data.reshape(count)


def load_mnist_training_set() -> tuple[np.ndarray, np.ndarray]:
    """Loads the normalized MNIST training split."""
    images = load_mnist_images(ensure_mnist_file("train-images-idx3-ubyte.gz"))
    labels = load_mnist_labels(ensure_mnist_file("train-labels-idx1-ubyte.gz"))
    if images.shape[0] != labels.shape[0]:
        raise ValueError("MNIST image and label counts do not match.")
    return images, labels


def select_balanced_mnist_subset(
    images: np.ndarray,
    labels: np.ndarray,
    samples_per_class: int,
    seed: int,
) -> dict[int, np.ndarray]:
    """Selects a reproducible balanced subset for each digit."""
    if samples_per_class < 2:
        raise ValueError("samples_per_class must be at least 2.")

    rng = np.random.default_rng(seed)
    by_digit: dict[int, np.ndarray] = {}

    for digit in range(10):
        indices = np.flatnonzero(labels == digit)
        if indices.size < samples_per_class:
            raise ValueError(f"Digit {digit} has only {indices.size} samples.")
        chosen = rng.choice(indices, size=samples_per_class, replace=False)
        by_digit[digit] = images[np.sort(chosen)]

    return by_digit


def polynomial_regression_example(lambda_reg: float) -> dict[str, Any]:
    """Builds and solves the feature-space example from Question 1.3."""
    x_samples = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=np.float64)
    targets = 1.0 + 0.5 * x_samples + x_samples**2
    design_matrix = build_polynomial_design_matrix(x_samples)
    primal_weights = solve_regularized_least_squares(design_matrix, targets, lambda_reg=lambda_reg)
    alpha, dual_weights = solve_kernelized_regularized_least_squares(
        design_matrix,
        targets,
        lambda_reg=lambda_reg,
    )
    predictions = design_matrix @ primal_weights

    return {
        "lambda_reg": float(lambda_reg),
        "x_samples": x_samples.tolist(),
        "targets": targets.tolist(),
        "design_matrix": design_matrix.tolist(),
        "estimated_weights": primal_weights.tolist(),
        "predictions": predictions.tolist(),
        "max_abs_primal_dual_difference": float(np.max(np.abs(primal_weights - dual_weights))),
        "alpha_norm": float(np.linalg.norm(alpha)),
    }


def mean_embedding_demo() -> dict[str, Any]:
    """Provides a concrete numerical check for Question 2.1."""
    x_samples = np.array(
        [
            [0.0, 1.0],
            [1.0, 0.0],
            [2.0, 1.0],
        ],
        dtype=np.float64,
    )
    z_query = np.array([1.0, 2.0], dtype=np.float64)
    sample_mean = np.mean(x_samples, axis=0)
    linear_eval = float(sample_mean @ z_query)
    direct_linear_eval = float(np.mean(x_samples @ z_query))

    gamma = 0.5
    rbf_values = np.exp(-gamma * np.sum((x_samples - z_query) ** 2, axis=1))
    rbf_eval = float(np.mean(rbf_values))

    return {
        "samples": x_samples.tolist(),
        "query": z_query.tolist(),
        "linear_sample_mean": sample_mean.tolist(),
        "linear_eval_via_mean": linear_eval,
        "linear_eval_direct_kernel_average": direct_linear_eval,
        "rbf_gamma": gamma,
        "rbf_eval_at_query": rbf_eval,
    }


def mnist_mmd_experiment(samples_per_class: int, seed: int) -> dict[str, Any]:
    """Computes pairwise RKHS discrepancies between MNIST digit classes."""
    images, labels = load_mnist_training_set()
    balanced = select_balanced_mnist_subset(
        images,
        labels,
        samples_per_class=samples_per_class,
        seed=seed,
    )

    pooled_for_bandwidth = np.vstack([balanced[digit][: min(20, samples_per_class)] for digit in range(10)])
    gamma = median_heuristic_gamma(pooled_for_bandwidth)

    pair_scores: list[PairScore] = []
    for digit_a in range(10):
        for digit_b in range(digit_a + 1, 10):
            x_class = balanced[digit_a]
            y_class = balanced[digit_b]
            k_xx = rbf_kernel_matrix(x_class, gamma=gamma)
            k_yy = rbf_kernel_matrix(y_class, gamma=gamma)
            k_xy = cross_rbf_kernel_matrix(x_class, y_class, gamma=gamma)
            pair_scores.append(
                PairScore(
                    digit_a=digit_a,
                    digit_b=digit_b,
                    mmd2_biased=biased_mmd2_from_gram(k_xx, k_yy, k_xy),
                )
            )

    pair_scores_sorted = sorted(pair_scores, key=lambda item: item.mmd2_biased)
    smallest_pairs = [asdict(item) for item in pair_scores_sorted[:5]]
    largest_pairs = [asdict(item) for item in pair_scores_sorted[-5:][::-1]]

    return {
        "samples_per_class": samples_per_class,
        "gamma": gamma,
        "smallest_pairs": smallest_pairs,
        "largest_pairs": largest_pairs,
    }


def kernel_geometry_sanity(gamma: float, seed: int) -> dict[str, Any]:
    """Numerically checks the identities used in Question 3."""
    images, labels = load_mnist_training_set()
    balanced = select_balanced_mnist_subset(images, labels, samples_per_class=5, seed=seed)
    x_small = np.vstack([balanced[0], balanced[1]])

    distances_formula = pairwise_squared_distances(x_small)
    distances_direct = np.sum((x_small[:, None, :] - x_small[None, :, :]) ** 2, axis=2)
    kernel_matrix = rbf_kernel_matrix(x_small, gamma=gamma)
    symmetrized = 0.5 * (kernel_matrix + kernel_matrix.T)
    eigenvalues = np.linalg.eigvalsh(symmetrized)

    return {
        "distance_formula_max_abs_error": float(np.max(np.abs(distances_formula - distances_direct))),
        "kernel_diagonal_max_abs_error": float(np.max(np.abs(np.diag(kernel_matrix) - 1.0))),
        "kernel_symmetry_max_abs_error": float(np.max(np.abs(kernel_matrix - kernel_matrix.T))),
        "kernel_min_eigenvalue": float(np.min(eigenvalues)),
    }


def write_results(output_path: Path, payload: dict[str, Any]) -> None:
    """Serializes the numeric outputs to JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2, sort_keys=True))


## 1. Regression in Feature Space and Kernel Formulation

### 1.1 Regularized least squares in feature space

We observe i.i.d. pairs

$$
\{(x_n,t_n)\}_{n=1}^N,
\qquad
x_n \in \mathbb{R}^P,
\qquad
t_n \in \mathbb{R},
$$

and the model

$$
t_n = \langle w,\phi(x_n)\rangle_{\mathcal H} + \eta_n,
\qquad
\eta_n \sim \mathcal N(0,\sigma_\eta^2).
$$

The unknown parameter is $w \in \mathcal H$. The regularized least-squares objective is

$$
\widehat w
=
\arg\min_{w \in \mathcal H}
\left[
\frac{1}{2}\sum_{n=1}^N
\bigl(t_n - \langle w,\phi(x_n)\rangle_{\mathcal H}\bigr)^2
+
\frac{\lambda}{2}\lVert w\rVert_{\mathcal H}^2
\right].
$$

The regularization parameter $\lambda$ penalizes large feature-space norms, stabilizes the inverse problem, and controls the bias-variance tradeoff.

### 1.2 Matrix derivation of the solution

Define the design matrix

$$
\Phi =
\begin{bmatrix}
\phi(x_1)^\top\\
\vdots\\
\phi(x_N)^\top
\end{bmatrix},
\qquad
t =
\begin{bmatrix}
t_1\\
\vdots\\
t_N
\end{bmatrix}.
$$

Then

$$
J(w) = \frac{1}{2}\lVert t-\Phi w\rVert_2^2 + \frac{\lambda}{2}\lVert w\rVert_2^2.
$$

Expanding and differentiating,

$$
\nabla_w J(w)
=
-\Phi^\top t + \Phi^\top \Phi w + \lambda w.
$$

Setting the gradient to zero yields

$$
(\Phi^\top \Phi + \lambda I)w = \Phi^\top t,
$$

so the estimator is

$$
\widehat w = (\Phi^\top \Phi + \lambda I)^{-1}\Phi^\top t.
$$

### 1.3 Polynomial feature map and implementation

For

$$
\phi(x)=
\begin{bmatrix}
1\\
x\\
x^2
\end{bmatrix},
$$

the design matrix becomes

$$
\Phi =
\begin{bmatrix}
1 & x_1 & x_1^2\\
1 & x_2 & x_2^2\\
\vdots & \vdots & \vdots\\
1 & x_N & x_N^2
\end{bmatrix}
\in \mathbb{R}^{N \times 3}.
$$

The next code cell runs a simple numerical example using

$$
x = [-2,-1,0,1,2]^\top,
\qquad
t = 1 + 0.5x + x^2.
$$


In [2]:
q1_results = polynomial_regression_example(lambda_reg=0.1)
print(json.dumps(q1_results, indent=2, sort_keys=True))


{
  "alpha_norm": 0.5226958631254532,
  "design_matrix": [
    [
      1.0,
      -2.0,
      4.0
    ],
    [
      1.0,
      -1.0,
      1.0
    ],
    [
      1.0,
      0.0,
      0.0
    ],
    [
      1.0,
      1.0,
      1.0
    ],
    [
      1.0,
      2.0,
      4.0
    ]
  ],
  "estimated_weights": [
    0.9673927749966176,
    0.49504950495049505,
    1.006629684751725
  ],
  "lambda_reg": 0.1,
  "max_abs_primal_dual_difference": 5.551115123125783e-17,
  "predictions": [
    4.003812504102528,
    1.4789729547978476,
    0.9673927749966176,
    2.4690719646988377,
    5.984010523904508
  ],
  "targets": [
    4.0,
    1.5,
    1.0,
    2.5,
    6.0
  ],
  "x_samples": [
    -2.0,
    -1.0,
    0.0,
    1.0,
    2.0
  ]
}


Interpretation:

- the estimated coefficients are very close to the generating polynomial $(1, 0.5, 1)$,
- the small mismatch is caused by the regularization term,
- the primal and dual solutions agree up to machine precision.

### 1.4 Kernel trick via the SVD

Let the thin singular value decomposition be

$$
\Phi = U_r \Sigma_r V_r^\top.
$$

Then

$$
\Phi^\top \Phi = V_r \Sigma_r^2 V_r^\top,
\qquad
\Phi \Phi^\top = U_r \Sigma_r^2 U_r^\top.
$$

The matrix

$$
K = \Phi \Phi^\top,
\qquad
K_{ij} = \langle \phi(x_i), \phi(x_j)\rangle_{\mathcal H},
$$

is the Gram matrix induced by the kernel. Defining

$$
\alpha = (K+\lambda I_N)^{-1}t,
$$

we get

$$
\widehat w = \Phi^\top \alpha,
$$

and therefore

$$
\widehat f(x)
=
\langle \widehat w,\phi(x)\rangle_{\mathcal H}
=
\sum_{n=1}^N \alpha_n k(x_n,x).
$$

This is the kernel trick: all computations are expressed through kernel evaluations and the Gram matrix, without explicitly constructing the feature map for prediction.


## 2. RKHS, Mean Embeddings, and Discrepancy Between Distributions

### 2.1 Empirical mean embedding

Let $k:\mathcal X \times \mathcal X \to \mathbb R$ be a positive-definite kernel with RKHS $\mathcal H_k$. For a sample $\{x_n\}_{n=1}^N$, the empirical mean embedding is

$$
\widehat \mu_X
=
\frac{1}{N}\sum_{n=1}^N \phi(x_n)
=
\frac{1}{N}\sum_{n=1}^N k(x_n,\cdot).
$$

For a query point $z$,

$$
\langle \widehat \mu_X, k(z,\cdot)\rangle_{\mathcal H_k}
=
\frac{1}{N}\sum_{n=1}^N k(x_n,z).
$$

For the linear kernel $k(x_i,x_j)=x_i^\top x_j$, the mean embedding is the ordinary sample mean in $\mathbb R^P$. For the Gaussian RBF kernel

$$
k(x_i,x_j)=\exp(-\gamma\lVert x_i-x_j\rVert_2^2),
$$

the RKHS is generally infinite-dimensional, but the embedding is still evaluated only through kernel averages.

### 2.2 Raw, centered, and standardized moments

In either the original space or the kernel-induced space:

- a **raw** moment is taken with respect to the origin,
- a **centered** moment is taken with respect to the mean,
- a **standardized** moment is centered and then scaled by the standard deviation.

In RKHS the relevant random object is $\phi(X)$ rather than $X$ itself, but the distinction is conceptually the same.

### 2.3 RKHS discrepancy between distributions

For distributions $P$ and $Q$ with embeddings

$$
\mu_P = \mathbb E[k(X,\cdot)],
\qquad
\mu_Q = \mathbb E[k(Y,\cdot)],
$$

the squared discrepancy is

$$
\lVert \mu_P - \mu_Q\rVert_{\mathcal H_k}^2.
$$

Expanding the norm gives the usual MMD expression:

$$
\operatorname{MMD}^2(P,Q)
=
\mathbb E[k(X,X')]
+
\mathbb E[k(Y,Y')]
-
2\mathbb E[k(X,Y)].
$$

For finite samples $X_m=\{x_i\}_{i=1}^m$ and $Y_n=\{y_j\}_{j=1}^n$, if

$$
(K_{XX})_{ij}=k(x_i,x_j),\quad
(K_{YY})_{ij}=k(y_i,y_j),\quad
(K_{XY})_{ij}=k(x_i,y_j),
$$

then the biased empirical estimator is

$$
\widehat{\operatorname{MMD}}_{\mathrm b}^2
=
\frac{1}{m^2}\mathbf 1_m^\top K_{XX}\mathbf 1_m
+
\frac{1}{n^2}\mathbf 1_n^\top K_{YY}\mathbf 1_n
-
\frac{2}{mn}\mathbf 1_m^\top K_{XY}\mathbf 1_n.
$$


In [3]:
q2_embedding = mean_embedding_demo()
q2_mmd = mnist_mmd_experiment(samples_per_class=100, seed=2026)

print(json.dumps({
    "mean_embedding_demo": q2_embedding,
    "mnist_mmd_experiment": q2_mmd,
}, indent=2, sort_keys=True))


{
  "mean_embedding_demo": {
    "linear_eval_direct_kernel_average": 2.3333333333333335,
    "linear_eval_via_mean": 2.333333333333333,
    "linear_sample_mean": [
      1.0,
      0.6666666666666666
    ],
    "query": [
      1.0,
      2.0
    ],
    "rbf_eval_at_query": 0.2903647218598325,
    "rbf_gamma": 0.5,
    "samples": [
      [
        0.0,
        1.0
      ],
      [
        1.0,
        0.0
      ],
      [
        2.0,
        1.0
      ]
    ]
  },
  "mnist_mmd_experiment": {
    "gamma": 0.009793297784779385,
    "largest_pairs": [
      {
        "digit_a": 0,
        "digit_b": 1,
        "mmd2_biased": 0.4686522687960123
      },
      {
        "digit_a": 1,
        "digit_b": 7,
        "mmd2_biased": 0.34204813526122635
      },
      {
        "digit_a": 1,
        "digit_b": 4,
        "mmd2_biased": 0.34160038410070204
      },
      {
        "digit_a": 1,
        "digit_b": 6,
        "mmd2_biased": 0.32930606221398184
      },
      {
        "digit_a": 1

In this run:

- the smallest RKHS discrepancies were `4 vs 9`, `3 vs 5`, `5 vs 8`, `7 vs 9`, and `8 vs 9`,
- the largest discrepancies were `0 vs 1`, `1 vs 7`, `1 vs 4`, `1 vs 6`, and `1 vs 9`.

A small MMD means the two classes induce similar kernel mean embeddings. A large MMD means the classes are well separated in the nonlinear geometry induced by the chosen RBF kernel.


## 3. Matrix Form of Kernels and Induced Geometry

For two samples $x_i, x_j \in \mathbb R^P$,

$$
\lVert x_i-x_j\rVert_2^2
=
(x_i-x_j)^\top(x_i-x_j)
=
x_i^\top x_i + x_j^\top x_j - 2x_i^\top x_j.
$$

If

$$
X =
\begin{bmatrix}
x_1^\top\\
\vdots\\
x_N^\top
\end{bmatrix},
\qquad
s =
\begin{bmatrix}
\lVert x_1\rVert_2^2\\
\vdots\\
\lVert x_N\rVert_2^2
\end{bmatrix},
$$

then the full matrix of squared distances is

$$
D = s\mathbf 1^\top + \mathbf 1 s^\top - 2XX^\top.
$$

For the Gaussian kernel

$$
K_{ij} = \exp(-\gamma \lVert x_i-x_j\rVert_2^2),
$$

all diagonal entries are constant because

$$
K_{ii} = \exp(0) = 1.
$$

Geometrically,

$$
K_{ii} = \lVert \phi(x_i)\rVert_{\mathcal H}^2 = 1,
$$

so every mapped sample lies on the unit sphere of the induced feature space.

The Gaussian kernel matrix is symmetric and positive semidefinite, so it is a Gram matrix in feature space.


In [4]:
q3_results = kernel_geometry_sanity(gamma=q2_mmd["gamma"], seed=2026)
print(json.dumps(q3_results, indent=2, sort_keys=True))


{
  "distance_formula_max_abs_error": 5.684341886080802e-14,
  "kernel_diagonal_max_abs_error": 5.551115123125783e-16,
  "kernel_min_eigenvalue": 0.0776883732159144,
  "kernel_symmetry_max_abs_error": 0.0
}


## Final Conclusion

The whole exam is organized around one idea: once inner products in feature space are replaced by kernel evaluations, we can pass seamlessly between

- regularized regression in feature space,
- dual/kernel regression through Gram matrices,
- mean embeddings and MMD in RKHS,
- Gaussian kernels as distance-based Gram matrices with a clear induced geometry.

This notebook is now the complete solution artifact for Exam 01.
